In [2]:
import requests
import pandas as pd
import time

url = "https://query.wikidata.org/sparql"

# Wikidata requires a User-Agent or it may reject the request
headers = {
    "Accept": "application/sparql-results+json",
    "User-Agent": "CityFreebaseIDFetcher/1.0 (your@email.com)"
}

def fetch_batch(offset=0, limit=5000):
    query = f"""
    SELECT DISTINCT ?cityLabel ?freebaseId ?iataCode ?countryLabel WHERE {{
      ?city wdt:P31/wdt:P279* wd:Q515.
      ?city wdt:P646 ?freebaseId.
      OPTIONAL {{ ?city wdt:P238 ?iataCode. }}
      OPTIONAL {{ ?city wdt:P17 ?country. }}
      SERVICE wikibase:label {{ bd:serviceParam wikibase:language "en". }}
    }}
    ORDER BY ?cityLabel
    LIMIT {limit}
    OFFSET {offset}
    """
    
    for attempt in range(3):  # retry up to 3 times
        try:
            r = requests.get(url, headers=headers, params={"query": query}, timeout=60)
            r.raise_for_status()
            
            if not r.text.strip():
                print(f"Empty response at offset {offset}, attempt {attempt+1}")
                time.sleep(5)
                continue
            
            data = r.json()["results"]["bindings"]
            return data
        except Exception as e:
            print(f"Error at offset {offset}, attempt {attempt+1}: {e}")
            time.sleep(10)
    
    return []

all_rows = []
offset = 0
limit = 5000

while True:
    print(f"Fetching offset {offset}...")
    batch = fetch_batch(offset, limit)
    
    if not batch:
        print("No more results or failed.")
        break
    
    for d in batch:
        all_rows.append({
            "city": d.get("cityLabel", {}).get("value", ""),
            "freebase_id": d.get("freebaseId", {}).get("value", ""),
            "iata_code": d.get("iataCode", {}).get("value", ""),
            "country": d.get("countryLabel", {}).get("value", ""),
        })
    
    print(f"  Got {len(batch)} rows. Total so far: {len(all_rows)}")
    
    if len(batch) < limit:
        break  # last page
    
    offset += limit
    time.sleep(2)  # be polite to Wikidata servers

df = pd.DataFrame(all_rows).drop_duplicates()
df.to_csv("cities_freebase_ids.csv", index=False)
print(f"Done. Saved {len(df)} cities to cities_freebase_ids.csv")

Fetching offset 0...
  Got 5000 rows. Total so far: 5000
Fetching offset 5000...
  Got 5000 rows. Total so far: 10000
Fetching offset 10000...
Error at offset 10000, attempt 1: HTTPSConnectionPool(host='query.wikidata.org', port=443): Read timed out. (read timeout=60)
Error at offset 10000, attempt 2: HTTPSConnectionPool(host='query.wikidata.org', port=443): Read timed out. (read timeout=60)
  Got 5000 rows. Total so far: 15000
Fetching offset 15000...
  Got 5000 rows. Total so far: 20000
Fetching offset 20000...
  Got 5000 rows. Total so far: 25000
Fetching offset 25000...
  Got 2490 rows. Total so far: 27490
Done. Saved 27490 cities to cities_freebase_ids.csv
